# Notebook 04: Checkpoint Sweep, Phase Detection & Adversarial Probes

**Prerequisites:** Notebook 03 must have completed.

**Outputs:** `experiments/results/sweep_*.npz`, `phase_transitions.json`, `adversarial_pre_post.npz`

In [ ]:
import sys; sys.path.insert(0, '..')
import json, numpy as np, torch
from pathlib import Path
from src.model.config import ModelConfig, EvalConfig
from src.model.train import load_pretrained_model, load_checkpoint, set_global_seed
from src.analysis.checkpoint_sweep import sweep_checkpoints, load_sweep_results
from src.analysis.phase_detection import detect_phase_transitions, summarise_transitions, detect_circuit_dissolution_step
from src.analysis.adversarial import compare_adversarial_pre_post

set_global_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
results_dir = Path('../experiments/results'); results_dir.mkdir(parents=True, exist_ok=True)
model_config = ModelConfig(); eval_config = EvalConfig()

In [ ]:
# Sweep code checkpoints (primary seed)
code_results = sweep_checkpoints(
    checkpoint_dir=Path('../checkpoints/code_seed42'),
    model_config=model_config, eval_config=eval_config,
    output_path=results_dir / 'sweep_code_seed42.npz', device=device,
)
print(f"Steps: {code_results['steps']}")
print(f"IS range: {code_results['induction_scores'].min():.3f} - {code_results['induction_scores'].max():.3f}")

In [ ]:
# Sweep prose checkpoints
prove_results = sweep_checkpoints(
    checkpoint_dir=Path('../checkpoints/prose_seed42'),
    model_config=model_config, eval_config=eval_config,
    output_path=results_dir / 'sweep_prose_seed42.npz', device=device,
)
print('Prose sweep done.')

In [ ]:
# Phase transition detection
trans_code = detect_phase_transitions(code_results['steps'], code_results['induction_scores'],
    threshold=eval_config.phase_threshold, window=eval_config.phase_window)
trans_prose = detect_phase_transitions(prove_results['steps'], prove_results['induction_scores'],
    threshold=eval_config.phase_threshold, window=eval_config.phase_window)
print('Code:', summarise_transitions(trans_code))
print('Prose:', summarise_transitions(trans_prose))
with open(results_dir / 'phase_transitions.json', 'w') as f:
    json.dump({'code': trans_code, 'prose': trans_prose}, f, indent=2)

In [ ]:
# Adversarial probes
model_pre = load_pretrained_model(model_config, device=device)
load_checkpoint(Path('../checkpoints/code_seed42/step_000000.pt'), model_pre)
model_post = load_pretrained_model(model_config, device=device)
last_ckpt = sorted(Path('../checkpoints/code_seed42').glob('step_*.pt'))[-1]
load_checkpoint(last_ckpt, model_post)
adv = compare_adversarial_pre_post(model_pre, model_post, model_pre.cfg.d_vocab, seq_len=20, batch_size=50, seed=42, device=device)
np.savez(results_dir / 'adversarial_pre_post.npz',
         pre=list(adv['pre'].values()), post=list(adv['post'].values()),
         delta=list(adv['delta'].values()), names=list(adv['pre'].keys()))
print('Top deltas:', sorted(adv['delta'].items(), key=lambda x: abs(x[1]), reverse=True)[:5])